In [1]:
# General imports 
import os 
import re
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as stats 
from skimage import measure
import matplotlib.pyplot as plt
import math

# Pynwb imports
from hdmf_zarr import NWBZarrIO
from nwbwidgets import nwb2widget

import sys
# Add the directory that contains the 'bci' folder
sys.path.insert(0, '/code/src/')  # Note: changed to /code/src/ not /code/src/bci/

# Now these imports should work
from bci.loaders import load
from bci.processing import processing

# Functions

In [2]:
def photostim_prepost_frames(epoch_table):
    epoch_table['duration'] = epoch_table.stop_time - epoch_table.start_time
    
    # Define the stimulus names we're looking for
    photo_stimuli = ['photostim', 'photostim_post']
    relevant_epochs = epoch_table[epoch_table['stimulus_name'].isin(photo_stimuli)].copy()
    relevant_epochs['duration'] = relevant_epochs.stop_time - relevant_epochs.start_time
    
    # Set stimulus_name as index for direct access
    indexed_epochs = relevant_epochs.set_index('stimulus_name')
    
    # Now you can directly access by stimulus name and convert to integers
    photostim_pre_start = int(indexed_epochs.loc['photostim'].start_frame)
    photostim_pre_stop = int(indexed_epochs.loc['photostim'].stop_frame)
    photostim_post_start = int(indexed_epochs.loc['photostim_post'].start_frame)
    photostim_post_stop = int(indexed_epochs.loc['photostim_post'].stop_frame)
    
    return photostim_pre_start, photostim_pre_stop, photostim_post_start, photostim_post_stop

In [3]:
def get_roi_centroids(valid_roi_table):
    '''
    Iterate through the image masks in the ROI table, 
    get the indices corresponding to each ROI, 
    then take the average to get the x and y locations of the cell centroid. 
    
    Returns an array with dimensions (n_cells, xy_position)
    '''
    centroids = []
    for mask in valid_roi_table['image_mask']:
        ys, xs = np.where(mask)
        x = np.mean(xs)
        y = np.mean(ys)
        centroids.append((x, y))
    return np.array(centroids)

In [4]:
def correlation_prepost_bci(dff_traces, photostim_pre_start, photostim_pre_stop, 
                           photostim_post_start, photostim_post_stop, valid_roi_table):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
    
    # Index into the dff traces array to limit to the relevant time points
    dff_pre = dff_traces[photostim_pre_start:photostim_pre_stop, :]
    dff_pre = dff_pre.T
    dff_post = dff_traces[photostim_post_start:photostim_post_stop, :]
    dff_post = dff_post.T
                            
    # Limit dff traces for photostimulation activity period before the task to valid ROIs
    dff_pre = dff_pre[valid_roi_table.index.values, :]
    print(f"dff_pre shape: {dff_pre.shape}")
    
    # Limit dff traces for photostimulation activity period after the task to valid ROIs
    dff_post = dff_post[valid_roi_table.index.values, :]
    print(f"dff_post shape: {dff_post.shape}")
    
    # Calculate correlation matrices
    correlations_pre = np.corrcoef(dff_pre)
    correlations_post = np.corrcoef(dff_post)
    
    # Keep only upper triangular part (avoid duplicate pairs and diagonal)
    correlations_pre = np.triu(correlations_pre, k=1)
    correlations_post = np.triu(correlations_post, k=1)
    
    # Statistical comparison
    t, p_val = stats.ttest_ind(correlations_pre.flatten(), correlations_post.flatten())
    print(f"T-test p-value: {p_val}")
    
    # Plot both distributions
    plt.figure(figsize=(10, 6))
    plt.hist(correlations_pre.flatten(), bins=100, color='green', label='pre', histtype='step')
    plt.hist(correlations_post.flatten(), bins=100, color='purple', label='post', histtype='step')
    plt.title('Correlations before and after BCI learning, p=' + str(np.round(p_val, 5)))
    plt.xlabel('Correlation')
    plt.ylabel('# cell pairs')
    plt.xlim(-0.1, 0.1)
    plt.legend()
    plt.show()
    
    return correlations_pre, correlations_post, p_val

In [7]:
def get_trial_correlations(trial_list, dff_traces):
    """Calculate correlation matrix for a set of trials"""
    all_trial_data = []


In [8]:
def correlation_prepost_stimulation(dff_traces_clean, pre_trials, post_trials, stimulated_neuron_id):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
    
    # Combine all pre-learning trials
    pre_data_list = []
    for _, trial in pre_trials.iterrows():
        trial_dff = dff_traces_clean[trial['start_frame']:trial['stop_frame'], :].T
        pre_data_list.append(trial_dff)
    dff_pre = np.concatenate(pre_data_list, axis=1)
    
    # Combine all post-learning trials  
    post_data_list = []
    for _, trial in post_trials.iterrows():
        trial_dff = dff_traces_clean[trial['start_frame']:trial['stop_frame'], :].T
        post_data_list.append(trial_dff)
    dff_post = np.concatenate(post_data_list, axis=1)
    
    print(f"dff_pre shape: {dff_pre.shape}")
    print(f"dff_post shape: {dff_post.shape}")
    
    # Calculate correlation matrices
    correlations_pre = np.corrcoef(dff_pre)
    correlations_post = np.corrcoef(dff_post)
    
    # Keep only upper triangular part (avoid duplicate pairs and diagonal)
    correlations_pre_upper = np.triu(correlations_pre, k=1)
    correlations_post_upper = np.triu(correlations_post, k=1)
    
    # Statistical comparison
    t, p_val = stats.ttest_ind(correlations_pre_upper.flatten(), correlations_post_upper.flatten())
    print(f"T-test p-value: {p_val}")
    
    # Plot both distributions
    plt.figure(figsize=(10, 6))
    plt.hist(correlations_pre_upper.flatten(), bins=100, color='green', label='pre', histtype='step')
    plt.hist(correlations_post_upper.flatten(), bins=100, color='purple', label='post', histtype='step')
    plt.title(f'Correlations before and after BCI learning (Neuron {stimulated_neuron_id} stimulation), p=' + str(np.round(p_val, 5)))
    plt.xlabel('Correlation')
    plt.ylabel('# cell pairs')
    plt.xlim(-0.1, 0.1)
    plt.legend()
    plt.show()
    
    return correlations_pre, correlations_post, p_val


# Carry on

In [9]:
metadata = load.load_filtered_metadata()

In [11]:
### Pulling the first session of the experiment
sesh_zero = metadata['name'].values[0]

In [12]:
data = load.load_session_data(sesh_zero)

BCI data directory: /data/brain-computer-interface

Session directory: /data/brain-computer-interface/single-plane-ophys_767715_2025-02-17_17-41-50_processed_2025-08-05_01-05-20

NWB file: single-plane-ophys_767715_2025-02-17_17-41-50_behavior_nwb
NWB path: /data/brain-computer-interface/single-plane-ophys_767715_2025-02-17_17-41-50_processed_2025-08-05_01-05-20/single-plane-ophys_767715_2025-02-17_17-41-50_behavior_nwb


/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.7.0 because version 2.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


All threshold files for mouse 767715: ['single-plane-ophys_767715_2025-02-10', 'single-plane-ophys_767715_2025-02-03', 'single-plane-ophys_767715_2025-01-31', 'single-plane-ophys_767715_2025-02-06', 'single-plane-ophys_767715_2025-02-17', 'single-plane-ophys_767715_2025-02-13']

Found threshold file at: /data/bci-thresholds/single-plane-ophys_767715_2025-02-17
total difference in dataframes: 4


In [13]:
data.keys()

dict_keys(['nwb_file', 'epoch_table', 'dff_traces', 'roi_table', 'frame_rate', 'bci_trials', 'thresholds'])

In [14]:
photostim = data['nwb_file'].stimulus["PhotostimTrials"].to_dataframe()

In [15]:
# Separate the two phases
photostim_trials = photostim[photostim['stimulus_name'] == 'photostim'].copy()
photostim_post_trials = photostim[photostim['stimulus_name'] == 'photostim_post'].copy()

print(f"Pre-learning trials: {len(photostim_trials)}")
print(f"Post-learning trials: {len(photostim_post_trials)}")

# Reset indices to match trial numbers
photostim_trials = photostim_trials.reset_index(drop=True)
photostim_post_trials = photostim_post_trials.reset_index(drop=True)

print("Ready to compare trial pairs!")

Pre-learning trials: 2064
Post-learning trials: 2140
Ready to compare trial pairs!


In [16]:
# Find neurons that were stimulated in both phases
pre_neurons = set(photostim_trials['closest_roi'])
post_neurons = set(photostim_post_trials['closest_roi'])
common_neurons = pre_neurons.intersection(post_neurons)

print(f"Neurons stimulated in both phases: {len(common_neurons)}")
print(f"Example neurons: {list(common_neurons)[:10]}")

Neurons stimulated in both phases: 100
Example neurons: [1, 2, 3, 4, 8, 11, 16, 17, 19, 23]


In [ ]:
# Run the analysis for all neurons
stimulated_neurons = list(common_neurons)
all_correlation_changes = {}

for i, target_neuron in enumerate(stimulated_neurons):
   print(f"Processing neuron {target_neuron} ({i+1}/{len(stimulated_neurons)})")
   
   pre_neuron_trials = pre_neurons[pre_trials['closest_roi'] == target_neuron]
   post_neuron_trials = post_trials[post_trials['closest_roi'] == target_neuron]
   
   correlations_pre, correlations_post, p_val = correlation_prepost_stimulation(
       dff_traces_clean, pre_neuron_trials, post_neuron_trials, target_neuron, show_plot=False)
   
   correlation_diff = correlations_post - correlations_pre
   upper_triangle = np.triu(correlation_diff, k=1)
   all_changes = upper_triangle[upper_triangle != 0]
   all_correlation_changes[target_neuron] = all_changes

# Create plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Combined histogram of all changes
all_changes_combined = np.concatenate(list(all_correlation_changes.values()))
ax1.hist(all_changes_combined, bins=100, alpha=0.7, color='gray')
ax1.set_xlabel('Correlation Change (Post - Pre)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of All Correlation Changes')
ax1.axvline(x=0, color='red', linestyle='--', alpha=0.7)

# Box plot for all neurons
data_for_boxplot = [changes for changes in all_correlation_changes.values()]
ax2.boxplot(data_for_boxplot)
ax2.set_xlabel('Stimulated Neuron Index')
ax2.set_ylabel('Correlation Change (Post - Pre)')
ax2.set_title('Distribution Summary: All Neurons')
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)

# Show fewer x-axis labels
neuron_ids = list(all_correlation_changes.keys())
ax2.set_xticks(range(0, len(neuron_ids), 10))
ax2.set_xticklabels([f'N{neuron_ids[i]}' for i in range(0, len(neuron_ids), 10)], rotation=45)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"Total correlation changes analyzed: {len(all_changes_combined)}")
print(f"Mean change: {np.mean(all_changes_combined):.4f}")
print(f"Median change: {np.median(all_changes_combined):.4f}")
print(f"Standard deviation: {np.std(all_changes_combined):.4f}")

In [ ]:
    
    for _, trial in trial_list.iterrows():
        start_frame = trial['start_frame']
        stop_frame = trial['stop_frame']
        trial_dff = dff_traces[start_frame:stop_frame, :].T  # Transpose so neurons are rows
        all_trial_data.append(trial_dff)
    
    # Concatenate all trials along time dimension
    combined_data = np.concatenate(all_trial_data, axis=1)
    
    # Calculate correlation matrix (neurons x neurons)
    correlation_matrix = np.corrcoef(combined_data)
    
    return correlation_matrix

# Calculate correlations for neuron 1 stimulation trials
pre_correlations = get_trial_correlations(pre_neuron_trials, dff_traces)
post_correlations = get_trial_correlations(post_neuron_trials, dff_traces)

print(f"Pre-learning correlation matrix shape: {pre_correlations.shape}")
print(f"Post-learning correlation matrix shape: {post_correlations.shape}")

# Calculate the difference
correlation_diff = post_correlations - pre_correlations
print(f"Correlation difference matrix shape: {correlation_diff.shape}")